In [ ]:
%reload_ext autoreload
%autoreload 2

from test.test_detections import test_match_points
import numpy as np
import matplotlib.pyplot as plt
from pose_estimator.utils.detections import polygon_to_obb
from pose_estimator.utils.detections import rotate_polygon, match_polygon_points

## Matching of image and object points for PnP


In [ ]:
image_points = [
    [899.64509704, 644.72216749],
    [923.958276, 651.17260491],
    [990.37261712, 400.84171179],
    [966.05943816, 394.39127437],
    [632.94149937, 431.29707767],
    [650.13228734, 435.15622909],
    [673.74082163, 329.99808228],
    [656.55003366, 326.13165085],
]

test_match_points(image_points)

In [ ]:
image_points = [
    [930.56243941, 516.40038511],
    [959.55463586, 520.93041563],
    [988.42022188, 336.19065775],
    [959.42802543, 331.66062724],
    [532.16492301, 588.50484504],
    [556.62337938, 591.58784384],
    [589.55873632, 330.30068717],
    [565.10027995, 327.21768837],
]

test_match_points(image_points)

In [ ]:
image_points = [
    [898.61701829, 415.81224583],
    [943.05918371, 518.8371991],
    [961.60104344, 510.83874464],
    [917.15887802, 407.81379136],
    [660.03728248, 486.17307214],
    [728.96537813, 689.32961919],
    [744.90223311, 683.92247342],
    [675.97413746, 480.76592637],
]

test_match_points(image_points)

## Computation of OBB for three points


In [ ]:
def test_obb(points):
    angle, obb = polygon_to_obb(points)
    obb = np.vstack((obb, obb[0]))  # Close the polygon for plotting
    points = np.vstack((points, points[0]))  # Close the polygon for plotting

    plt.plot(*points.T, "b-", label="Polygon Points")
    plt.plot(*obb.T, "r-", label="Minimum Rotated Rectangle")
    plt.gca().set_aspect("equal")
    plt.gca().invert_yaxis()
    plt.legend()
    plt.show()

    print(angle)

In [ ]:
points = np.array([(-500, 500), (200, 250), (150, 100), (50, 150)])
test_obb(points)

points = np.array([(-500, 250), (200, 250), (150, 100), (50, 150)])
test_obb(points)

points = np.array([(100, 200), (300, 250), (150, 100)])
test_obb(points)

points = np.array([(50, 500), (200, 250), (150, 100), (50, 150)])
test_obb(points)

## Rotate polygon


In [ ]:
points = np.array([(100, 200), (200, 250), (150, 100)])

angle = 135
rotated_points = rotate_polygon(points, angle)

points = np.vstack((points, points[0]))  # Close the polygon for plotting
rotated_points = np.vstack(
    (rotated_points, rotated_points[0])
)  # Close the polygon for plotting

plt.plot(*points.T, "g-", label="Polygon")
plt.plot(*rotated_points.T, "m-", label="Rotated Polygon")
plt.gca().set_aspect("equal")
plt.gca().invert_yaxis()

## Matching for Bin


In [ ]:
# Problematic
points = np.array(
    [
        [664.82985317, 552.58952676],
        [1410.63442629, 939.72480708],
        [1205.01063459, 1335.85291007],
        [459.20606147, 948.71762975],
    ]
)

# Problematic
points = np.array(
    [
        [664.82985317, 552.58952676],
        [1410.63442629, 939.72480708],
        [1205.01063459, 1335.85291007],
        [459.20606147, 948.71762975],
    ]
)

# points = np.array(
#     [
#         [1352.73780783, 543.56664614],
#         [1665.03251787, 817.65509959],
#         [1090.3749704, 1472.41639984],
#         [778.08026036, 1198.32794639],
#     ]
# )

object_points = np.array(
    [[0.0, 0.0], [0.30479997, 0.0], [0.30479997, 0.60959995], [0.0, 0.60959995]]
)

# Match image points and object points
angle, detected_points = polygon_to_obb(points)

# Rotate object points before matching by point distances because bin yaw can
# be large and we assume perspective does not change imaged aspect ratio by much.
object_points, image_points = match_polygon_points(
    object_points, detected_points, A_angle=angle
)

print("Angle:", angle)


# Plot as image coordinates (origin at top-left)
plt.figure(figsize=(6, 6))
plt.scatter(points[:, 0], points[:, 1], c="red", marker="o")

# Annotate points with indices
for i, (x, y) in enumerate(image_points):
    plt.text(x, y, f"{i+1}", color="blue", fontsize=12)

plt.gca().invert_yaxis()  # Y-axis downwards like image coordinates
plt.xlabel("X (pixels)")
plt.ylabel("Y (pixels)")
plt.title("2D Points in Image Coordinates")
plt.grid(True)
plt.axis("equal")
plt.show()

In [ ]:
rotated_object_points = rotate_polygon(object_points, angle)
plt.plot(*rotated_object_points.T, "g-", label="Object Points")

# Annotate points with indices
for i, (x, y) in enumerate(rotated_object_points):
    plt.text(x, y, f"{i+1}", color="blue", fontsize=12)

plt.gca().invert_yaxis()  # Y-axis downwards like image coordinates
plt.xlabel("X (pixels)")
plt.ylabel("Y (pixels)")
plt.title("2D Points in Image Coordinates")
plt.grid(True)
plt.axis("equal")
plt.show()